# C3 · 캡스톤 3: MCMC와 이징 모형 (Capstone 3: MCMC and the 2D Ising Model)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §1.6 (세부 균형·Metropolis–Hastings), §1.5 (극한 거동) · Lawler 2e Ch. 7 (가역 사슬·MCMC) · Norris §1.9 (시간 역전), §5.5 (MCMC) · 추가: Levin–Peres–Wilmer 2e §3.3, §12.2, Ch. 15 (Glauber 동역학·스펙트럴 갭·이징) · Newman & Barkema Ch. 3–4 (이징 메트로폴리스·자기상관·Wolff) |
| 선수 노트북 | 01f (메트로폴리스·세부 균형·`metropolis_chain`), 01e (TV 거리·$\lambda_2$·수렴 정리), 05a (정상열의 분산·조건부 기대), 01d (`stationary`), 05r, 00b (SE·반복) |
| 예상 소요 | 150분 |
| 상태 | draft |

## 프로젝트 브리프 · 합격 기준 · 단계

**브리프.** 2차원 이징 모형(2D Ising model; $J=1$, $k_B=1$, 외부장 $h=0$, 주기 경계)을 메트로폴리스 MCMC로 표본화하여 (i) 자화 곡선 $\langle|m|\rangle(T)$와 Onsager 임계온도 $T_c = 2/\ln(1+\sqrt2) = 2.26919$를 재현하고, (ii) 세부 균형·정상분포를 완전 열거가 가능한 $3\times3$(512 상태)에서 **정확히** 검증하며(01f), (iii) 스펙트럴 갭과 TV 거리로 혼합 속도를 보고(01e), (iv) 자기상관 시간 $\tau_{int}$의 임계 감속(critical slowing down)을 측정한다(05a의 정상열 분산). 격자 크기: FAST $L=16$(9개 온도, 4,000 sweep), full $L=32$(13개 온도, 20,000 sweep) + 임계 감속용 $L\in\{8,16,32\}$.

**합격 기준 — 이 그림/숫자가 나오면 성공.**

1. $3\times3$ 메트로폴리스 전이행렬 $P$(spkit `metropolis_chain`)가 볼츠만 분포를 정상분포로 갖고 세부 균형을 $10^{-12}$ 이내로 만족한다. $T_c$에서 $|\lambda_2| = 0.996348$(홀 모드), $|\lambda_3| = 0.944340$(짝 모드).
2. 균등 초기분포에서 TV 거리가 $|\lambda_3|^n$을 따라 $n=300$에 $3.6\times10^{-8}$까지 떨어지지만, all-up 초기상태에서는 $n=300$에도 $0.176$(느린 홀 모드) — 두 곡선이 한 그림에.
3. `simulate_chain`으로 만든 $3\times3$ 표본의 $\langle|m|\rangle$, $\langle e\rangle$, $\langle m^2\rangle$이 정확값 $0.87107$, $-1.61125$, $0.80943$과 4 SE 이내.
4. 체커보드 메트로폴리스 샘플러의 $4\times4$ 결과가 완전 열거 정확값($T_c$: $\langle|m|\rangle = 0.84386$, $\langle e\rangle = -1.56562$; $T=2.5$: $0.76471$, $-1.37912$)과 4 SE 이내.
5. $L=16$/$32$ 자화 곡선이 $T<2.1$에서 Onsager 곡선과 겹치고 $T_c$ 근처에서 급락하며($L=16$: $T_c$에서 $\approx0.72$, $L=32$: $\approx0.64$), $\tau_{int}(|m|)$이 $T_c$ 근방에서 봉우리를 이루고 $L$과 함께 커진다($L=8$: ≈3, 16: ≈9, 32: ≈40 sweep).

**단계.**

| 단계 | 내용 | 재사용 | 산출물 |
|---|---|---|---|
| Task 1 | $3\times3$ 정확 사슬: 열거, `metropolis_chain`, 세부 균형, 스펙트럼, TV 곡선 | 01f, 01e, 01d | 합격 1·2, TV 그림 |
| Task 2 | $3\times3$ 사슬을 `simulate_chain`으로 표본화해 정확값과 대조 | 00b | 합격 3, 히스토그램 |
| Task 3 | 체커보드 샘플러 구현 + $4\times4$ 완전 열거 검증 | 01f | 합격 4 |
| Task 4 | 자화 곡선 $\langle|m|\rangle, \langle e\rangle, \chi, C$ vs Onsager | — | 합격 5(앞부분), 2×2 그림 |
| Task 5 | 자기상관 시간과 임계 감속 | 05a, 00b | 합격 5(뒷부분), $\tau(T)$ 그림 |

스트레치 목표(Binder 누적률, Wolff 군집, 유한 크기 스케일링)는 연습문제 S1–S3에 있다.

## 0. 준비 (Setup)

In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")  # 512×512 고유값 분해가 다중 스레드 BLAS에서 10배 느려지는 것을 막음 (numpy import 전에!)
import time
import itertools
import numpy as np
import scipy.linalg as sla
import matplotlib.pyplot as plt
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import metropolis_chain, is_stochastic, stationary, tv_distance, simulate_chain
from spkit.plots import plot_hist_vs_pmf

SEED, rng = rng_for("C3")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
T_C = 2 / np.log(1 + np.sqrt(2))   # Onsager 임계온도 (J = k_B = 1)
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, T_c={T_C:.9f}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (05r/05a) 마팅게일의 정의를 쓰고, 조건부 기대의 tower property가 $E[M_{n+k}\mid\mathcal F_n] = M_n$을 어떻게 주는지 설명하라.

<details><summary>정답</summary>

$E|M_n|<\infty$, $M_n$이 $\mathcal F_n$-가측, $E[M_{n+1}\mid\mathcal F_n] = M_n$. tower: $E[M_{n+2}\mid\mathcal F_n] = E[E[M_{n+2}\mid\mathcal F_{n+1}]\mid\mathcal F_n] = E[M_{n+1}\mid\mathcal F_n] = M_n$, 이하 귀납.

</details>

**Q2.** (05r/05d) 마팅게일 수렴 정리의 조건은 무엇이며 폴리아 항아리의 비율은 무엇으로 수렴하는가?

<details><summary>정답</summary>

$\sup_n E|M_n| < \infty$($L^1$ 유계; 예: 음이 아닌 마팅게일)이면 $M_n \to M_\infty$ a.s. 폴리아 항아리(처음 $a$ 흰, $b$ 검)의 흰 공 비율은 Beta$(a,b)$ 분포를 갖는 극한으로 수렴한다.

</details>

**Q3.** (05r/04c) $\pi(x) = 1/E_x[T_x^+]$를 재생–보상 정리로 어떻게 증명하는가?

<details><summary>정답</summary>

$x$ 방문 시각들이 재생 과정을 이룬다. 사이클당 보상 1(방문 1회)이므로 장기 방문 비율 $= 1/E_x[T_x^+]$; 에르고딕 정리로 이 비율이 $\pi(x)$.

</details>

**Q4.** (05c, 두 노트북 전) 정상(stationary) 열 $f(X_0), f(X_1), \dots$의 표본평균 분산이 왜 $\sigma^2/n$이 아닌가? 어떤 식이 맞는가?

<details><summary>정답</summary>

항들이 상관되어 있다. $\mathrm{Var}(\bar f_n) \approx \frac{\sigma^2}{n}\big(1 + 2\sum_{k\ge1}\rho_k\big) = \frac{\sigma^2}{n}\cdot2\tau_{int}$, 유효 표본 수는 $n/(2\tau_{int})$. 이 노트북의 Task 5가 바로 이 보정이다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **2D 이징 모형(Ising model).** $L\times L$ 토러스(주기 경계)의 스핀 $s_i\in\{-1,+1\}$, 에너지 $H(s) = -J\sum_{\langle ij\rangle}s_is_j$ (최근접 쌍, 각 결합 한 번씩; $N=L^2$개 사이트에 $2N$개 결합), $J=1$, $h=0$. **볼츠만 분포(Boltzmann distribution)** $\pi_\beta(s) = e^{-\beta H(s)}/Z_\beta$, $\beta = 1/T$. $Z_\beta$는 $2^N$개 항의 합이라 큰 $L$에서는 계산 불가 — MCMC가 필요한 이유다.
- **관측량(스핀당).** 자화 $m = \frac1N\sum_i s_i$, 에너지 $e = H/N$ (all-up: $e=-2$). $\langle|m|\rangle$, $\langle e\rangle$, 자화율(susceptibility) $\chi = N\beta(\langle m^2\rangle - \langle|m|\rangle^2)$, 비열(specific heat) $C = N\beta^2(\langle e^2\rangle-\langle e\rangle^2)$, Binder 누적률 $U = 1 - \langle m^4\rangle/(3\langle m^2\rangle^2)$.
- **메트로폴리스 단일 스핀 뒤집기.** 사이트 $i$를 제안, $\Delta E = 2Js_i\sum_{j\sim i}s_j \in\{-8,-4,0,4,8\}$, 수용확률 $\min(1, e^{-\beta\Delta E})$. **random-scan**: 사이트를 균등 무작위로 고름($3\times3$ 정확 사슬); **checkerboard-scan**: 짝 부분격자 전체를 동시에 갱신한 뒤 홀 부분격자 전체(벡터화 샘플러). 1 sweep = $N$번 제안.
- **Onsager.** $T_c = 2/\ln(1+\sqrt2) = 2.269185314$; 자발 자화 $m(T) = \big(1 - \sinh^{-4}(2/T)\big)^{1/8}$ ($T<T_c$), $0$ ($T\ge T_c$). $m(1.5) = 0.98650$, $m(2.0) = 0.91132$, $m(2.2) = 0.78476$, $m(2.25) = 0.67185$.
- **적분 자기상관 시간(integrated autocorrelation time).** $\tau_{int} = \frac12 + \sum_{k\ge1}\rho_k$ (sweep 단위; Sokal 창 $M \ge c\tau$, $c=6$으로 절단). $\mathrm{Var}(\bar x) \approx \sigma^2\cdot2\tau_{int}/n$, 유효 표본 수 $n_{eff} = n/(2\tau_{int})$. **임계 감속**: $\tau \sim \xi^z$, $T_c$에서 상관 길이 $\xi\sim L$이므로 $\tau\sim L^z$ (메트로폴리스 $z\approx2.17$).
- **전역 반전** $F: s\mapsto -s$. $H(Fs) = H(s)$이므로 $\pi_\beta$와 $P$가 $F$-불변; 관측량은 짝($|m|, m^2, e$) 또는 홀($m$)로 나뉜다.

### 2.2 정리 1 — 메트로폴리스 갱신의 세부 균형 (01f; Durrett 3e §1.6)

> 대칭 제안 $q(x,y)=q(y,x)$와 수용 $\alpha(x,y) = \min(1,\pi(y)/\pi(x))$로 만든 $P(x,y) = q(x,y)\alpha(x,y)$ ($x\ne y$)는 $\pi(x)P(x,y) = \pi(y)P(y,x)$를 만족하고 따라서 $\pi P = \pi$. 이징에서 $\pi(y)/\pi(x) = e^{-\beta\Delta E}$이므로 $Z$를 몰라도 된다.

**가정이 왜 필요한가**

- 제안의 대칭성: 아니면 Hastings 보정 $q(y,x)/q(x,y)$가 필요하다(spkit `metropolis_chain`은 일반형이라 보정을 자동으로 넣는다).
- 기약성은 별도의 가정이다: 모든 단일 뒤집기의 수용확률이 $>0$($\beta<\infty$)이므로 임의 두 상태가 연결된다; 비주기성은 기각(자기 루프) 확률 $>0$에서 온다.

*증명 (인용).* 01f에서 완전 증명(Durrett 3e §1.6). 여기서는 Task 1에서 $512\times512$ 행렬로 수치 확인한다($\max|\pi_iP_{ij}-\pi_jP_{ji}| < 10^{-14}$).

### 2.3 정리 2 — 체커보드(병렬) 갱신의 타당성

> $L$이 짝수이면 같은 색 부분격자의 사이트들은 서로 이웃이 아니다. 짝 부분격자의 단일 사이트 메트로폴리스 커널 $K_i$들은 서로 가환이고 각각 $\pi$-가역이므로 곱 $K_{even} = \prod_{i\in even}K_i$는 $\pi$-가역이다. 마찬가지로 $K_{odd}$. 합성 $K = K_{even}K_{odd}$는 $\pi$를 정상분포로 갖는다: $\pi K = (\pi K_{even})K_{odd} = \pi K_{odd} = \pi$. (단, $K$ 자체는 일반적으로 $\pi$-가역이 아니다.)

**가정이 왜 필요한가**

- 같은 색 사이트가 비인접: 사이트 $i$의 $\Delta E$가 반대 색 이웃에만 의존하므로 짝 사이트들을 동시에 갱신해도 각 $K_i$의 전이확률이 변하지 않는다(가환성). 이웃을 동시에 갱신하면(E3의 '전체 병렬') 곱 표현이 깨지고 정상분포가 볼츠만이 아니게 된다 — $T=2.0$에서 $\langle|m|\rangle\approx0$, $\langle e\rangle\approx+1.96$(반강자성 상태)이라는 완전히 틀린 답이 나온다.
- $L$ 짝수: 홀수면 주기 경계에서 체커보드 색이 맞지 않아 같은 색 이웃이 생긴다.
- $0<\beta<\infty$: $\beta=0$이면 모든 제안이 수용되어 체커보드 갱신이 결정론적 전역 반전이 되고($|m|$ 고정) 에르고딕성이 깨진다(E3).

<details><summary>증명</summary>

(1) *$K_i$의 $\pi$-가역성.* 사이트 $i$만 제안하는 커널 $K_i(x, x^{(i)}) = \min(1, \pi(x^{(i)})/\pi(x))$, $K_i(x,x) = 1 - K_i(x,x^{(i)})$ ($x^{(i)}$는 $i$만 뒤집은 상태)는 정리 1(제안이 $x\leftrightarrow x^{(i)}$ 대칭)에 의해 $\pi$-가역이다.

(2) *가환성.* $i, j$가 같은 색이면 $j\notin\partial i$이다. $K_i$는 $s_j$를 바꾸지 않고, $K_i$의 전이확률은 $s_i$와 $\partial i$의 스핀에만 의존하므로 $s_j$에 의존하지 않는다. 그러므로 $K_iK_j$와 $K_jK_i$ 모두 "$i$를 (현재 이웃을 보고) 뒤집을지 결정, $j$를 (현재 이웃을 보고) 독립적으로 결정"과 같은 곱 커널이다: $K_iK_j = K_jK_i$.

(3) *가역 커널의 가환 곱은 가역.* $K_i, K_j$가 $\pi$-가역이고 가환이면
$$\pi(x)(K_iK_j)(x,y) = \sum_z\pi(x)K_i(x,z)K_j(z,y) = \sum_z K_i(z,x)\pi(z)K_j(z,y) = \sum_z K_i(z,x)K_j(y,z)\pi(y) = \pi(y)(K_jK_i)(y,x) = \pi(y)(K_iK_j)(y,x).$$
귀납으로 $K_{even} = \prod_{i\in even}K_i$가 $\pi$-가역이고, 같은 논법으로 $K_{odd}$도 그렇다.

(4) *정상성의 합성.* 가역이면 정상: $\pi K_{even} = \pi$, $\pi K_{odd} = \pi$. 따라서 $\pi K = (\pi K_{even})K_{odd} = \pi K_{odd} = \pi$.

(5) *비가역성 주의.* $K_{even}K_{odd} \ne K_{odd}K_{even}$이므로(짝 사이트의 결정은 홀 이웃의 *갱신 전* 값을 본다) $K$의 고유값은 복소수일 수 있으나, 대칭 스킴 $K_{even}K_{odd}K_{even}$은 가역이다. 기약·비주기성은 정리 1의 비고와 같다: $0<\beta<\infty$이면 모든 단일 뒤집기가 양의 확률을 갖고 기각도 양의 확률이다. $\blacksquare$

</details>

### 2.4 정리 3 — 반전 대칭과 홀/짝 고유모드: 느린 혼합 vs 빠른 관측량 수렴 (LPW 2e §12.2 참고)

> $P$가 $F$와 가환이면($P(Fx,Fy) = P(x,y)$) 가역 사슬의 고유벡터를 $F$-짝 또는 $F$-홀로 고를 수 있다. (a) 초기분포 $\mu$가 $F$-대칭($\mu\circ F = \mu$)이면 $\mu P^n - \pi$는 짝 모드만 포함해 $\|\mu P^n-\pi\| \le C|\lambda_{2,even}|^n$; (b) 짝 관측량 $f$($f\circ F = f$)에 대해서는 임의의 시작에서 $E_x[f(X_n)] - \pi(f) = \sum_{k\ \mathrm{even}}\lambda_k^n(\cdots)$ — 홀 모드는 기여하지 않는다. $3\times3$, $T_c$: 가장 느린 모드 $\lambda_2 = 0.996348$은 홀(두 자화 상 사이의 터널링), 가장 느린 짝 모드는 $\lambda_3 = 0.944340$.

**가정이 왜 필요한가**

- 가역성: $\ell^2(\pi)$에서 고유벡터가 직교 완비계를 이루어 전개가 가능하다(비가역이면 조르당 블록이 생길 수 있다).
- $F$-가환성: $H$의 반전 불변성($h=0$)에서 온다. 외부장 $h\ne0$이면 대칭이 깨져 홀/짝 분리가 없다.

*증명 (스케치).* $F$는 대합이고 $P$와 가환하며 $\pi$를 보존하므로 $\ell^2(\pi)$에서 자기수반 유니타리이다; $P$의 각 고유공간을 $F$의 $\pm1$ 고유공간으로 분해할 수 있다. 오른쪽 고유함수 $r_k$($\ell^2(\pi)$ 정규직교, $r_1\equiv1$)로 전개하면 $\mu P^n = \pi + \sum_{k\ge2}\lambda_k^n\langle\mu/\pi, r_k\rangle_\pi\,(\pi r_k)$; $\mu$가 $F$-대칭이면 홀 $r_k$와의 내적이 0이다. (b)는 $E_x[f(X_n)] = \sum_k\lambda_k^n r_k(x)\langle f, r_k\rangle_\pi$에서 홀 $r_k$와 짝 $f$의 내적이 0. 수치($3\times3$, $T_c$): 균등 시작 TV: $n=50$: $5.9\times10^{-2}$, $100$: $3.4\times10^{-3}$, $300$: $3.6\times10^{-8}$ ($\approx|\lambda_3|^n$); all-up 시작 TV: $0.440, 0.367, 0.176$ ($\approx$ 상수 $\times|\lambda_2|^n$). all-up에서 홀 관측량 $E[m(X_n)]$: $n=100$: $0.673$, $200$: $0.467$, $300$: $0.324$ (비율 $0.694 = \lambda_2^{100}$).

### 2.5 정리 4 — Onsager (1944), Yang (1952): 2D 이징의 정확해

> $\sinh(2\beta_cJ) = 1$, 즉 $T_c = 2J/\ln(1+\sqrt2)$; $T<T_c$에서 자발 자화 $m(T) = (1-\sinh^{-4}(2\beta J))^{1/8}$, $T\ge T_c$에서 $0$. 유한 $L$에서는 $\langle|m|\rangle_L > m(T)$이고 $L\to\infty$에서 수렴하며, $T_c$에서 $\langle|m|\rangle_L \sim L^{-1/8}$.

**가정이 왜 필요한가**

- 열역학적 극한 $L\to\infty$: 유한 $L$에서는 $Z$가 $\beta$의 해석함수라 상전이가 없다 — 그림의 '급락'이 날카롭지 않은 이유.
- $h=0$, 최근접 상호작용, 정사각 격자: Onsager 해의 조건.

*증명 (인용).* 전달행렬 대각화(Onsager 1944), 자화는 Yang 1952; Newman & Barkema Ch. 3에 결과 인용. 본 노트북은 수치 대조만 한다: $L=32$, $T=2.0$: $0.911$ vs $0.9113$; $T=2.2$: $0.789$ vs $0.7848$; $T_c$: $0.64$ vs $0$(유한 크기).

### 2.6 정리 5 — 자기상관과 표본평균의 분산 (05a 연계)

> 정상 열 $x_t$의 표본평균 분산은 $\mathrm{Var}(\bar x_n) \approx \frac{\sigma^2}{n}\,2\tau_{int}$, $\tau_{int} = \frac12+\sum_{k\ge1}\rho_k$. 독립 반복(replica) 평균들의 표본분산으로 구한 SE는 이 보정을 자동으로 포함한다.

**가정이 왜 필요한가**

- 정상성(워밍업 후)과 $\sum|\rho_k|<\infty$: 스펙트럴 갭이 있으면 $\rho_k$는 기하급수적으로 감쇠한다.
- Sokal 창: 무한합의 추정은 노이즈가 누적되므로 $M\ge c\tau$에서 절단한다.

*증명 (스케치).* $\mathrm{Var}(\sum_{t<n}x_t) = \sum_{s,t}\mathrm{Cov}(x_s,x_t) = n\sigma^2\big(1 + 2\sum_{k=1}^{n-1}(1-k/n)\rho_k\big) \to n\sigma^2\cdot2\tau_{int}$. 본 노트북에서는 $4\times4$ 검증에 $R=200$ 독립 복제를 써서 SE를 얻고(Task 3), 큰 격자에서는 $\tau_{int}$로 SE를 보정한다(Task 4·5).

### 2.7 직관

MCMC는 '목표분포를 정상분포로 갖는 사슬을 설계해 오래 돌리는 것'이고, 이징은 그 사슬의 병리(느린 혼합)가 물리(상전이)와 정확히 겹치는 예다. 낮은 $T$에서는 두 자화 상(all-up/all-down) 사이의 터널링이 거의 불가능해 홀 모드가 느리지만, 우리가 원하는 $|m|$은 짝 관측량이라 그 느림을 못 느낀다 — 대신 $T_c$ 근처에서는 상관 길이가 격자만큼 커져 짝 모드까지 느려진다(임계 감속). $3\times3$의 $512\times512$ 행렬은 이 모든 것을 고유값으로 보여 주는 현미경이다.

### 2.8 함정 (traps)

- 유한 격자에서 $\langle|m|\rangle$는 $T_c$ 위에서도 0이 아니다($L=16$, $T=3$: $0.17$; $T=\infty$ 극한에서도 $E|m| = 0.0498$ for $N=256$). 상전이는 $L\to\infty$에서만 날카롭다.
- 부호 있는 $\langle m\rangle$은 대칭으로 정확히 0이다 — 자화를 $|m|$로 재야 한다. all-up에서 시작하면 $E[m(X_n)]$이 $\lambda_2^n$으로 아주 천천히 0으로 가므로($3\times3$에서 300 step 뒤 $0.32$) '자화가 남아 있다'고 착각하기 쉽다.
- TV 거리가 큰 것과 관측량이 틀린 것은 다르다: all-up 시작 $3\times3$의 TV는 300 step 뒤 $0.18$이지만 $\langle|m|\rangle$·$\langle e\rangle$는 이미 정확하다(정리 3(b)).
- sweep 단위 표본으로 iid인 양 SE를 구하면 $\sqrt{2\tau_{int}}$배 과소평가된다; 독립 복제 또는 $\tau_{int}$ 보정을 써라.
- 체커보드 갱신은 이웃을 동시에 갱신하지 않기 때문에 옳다. '전부 동시에'로 바꾸면 반강자성 인공물($\langle e\rangle>0$)이 나온다(E3). $\beta=0$에서는 체커보드 자체가 결정론적으로 퇴화한다.
- $3\times3$ 주기 경계는 각 사이트가 4개 이웃을 갖지만 $2\times2$는 결합이 중복된다 — 완전 열거 검증에 $L=2$를 쓰지 말 것.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

## 3. Predict — 코드를 돌리기 전에 예측

숫자를 먼저 적어 보세요. 정답은 Task와 Compare 표에서 계산됩니다(여기에 적지 마세요).

In [ ]:
predictions = {
    "T_c": None,                  # 2D 이징의 임계온도 T_c (J = k_B = 1)는? 자화 곡선이 어디서 급락할까?
    "m_onsager_T2": None,         # T = 2.0에서 Onsager 자발 자화 m(T)는? L=32 시뮬레이션의 <|m|>는 이보다 클까 작을까?
    "absm_3x3_Tc": None,          # 3×3 토러스, T = T_c에서 정확한 <|m|>는? (0에 가까울까?)
    "lambda2_lambda3_3x3": None,  # 3×3 random-scan 사슬(T_c)의 (|λ2|, |λ3|) 튜플은? 어느 것이 홀 모드일까?
    "acc_rate_3x3_Tc": None,      # 3×3, T_c에서 정상상태 제안 수용률은?
    "absm_L16_Tc": None,          # L = 16, T = T_c의 <|m|>는 대략? (L = 32에서는?)
    "tau_Tc": None,               # T_c에서 |m|의 τ_int(sweep)은 L = 16에서 대략? (T = 1.5에서는?)
}

## 4. 단계별 작업 (Tasks) — Simulate

### Task 1 — $3\times3$ 정확 사슬: 열거, `metropolis_chain`, 세부 균형, 스펙트럼, TV 곡선 (01f/01e 재사용)

512개 상태를 열거하고 $E$, $M = \sum_i s_i$를 계산한다. 상태 $k$의 비트 표현을 그대로 쓰면 '사이트 $i$ 뒤집기 = 비트 XOR'이므로 제안행렬과 전역 반전 순열 $F$가 한 줄씩이다. 목표분포는 정규화하지 않은 `exp(-beta*(E - E.min()))`로 넘겨도 된다 — 메트로폴리스는 비율만 본다.

In [ ]:
def ising_energy(s):
    """H(s) = -Σ_<ij> s_i s_j (주기 경계, 결합 한 번씩). s: (..., L, L)."""
    return -(s * np.roll(s, 1, -1) + s * np.roll(s, 1, -2)).sum(axis=(-1, -2))

def boltzmann(E, T):
    w = np.exp(-(E - E.min()) / T)      # E.min()을 빼면 overflow 없음(정규화에 영향 없음)
    return w / w.sum()

# 3×3: 512 상태. 상태 k의 사이트 i 스핀 ↔ k의 비트 (MSB = 사이트 0, 비트 1 ↔ +1)
states3 = np.array(list(itertools.product([-1, 1], repeat=9)), dtype=np.int8).reshape(-1, 3, 3)
E3, M3 = ising_energy(states3), states3.sum(axis=(1, 2))
m3 = M3 / 9
codes = ((states3.reshape(-1, 9) == 1) << np.arange(8, -1, -1)).sum(1)
assert np.array_equal(codes, np.arange(512))   # itertools 순서가 비트 순서와 같음을 확인
prop3 = np.zeros((512, 512))
for i in range(9):                              # 사이트 i 뒤집기 = 비트 (8-i) XOR, 제안확률 1/9
    prop3[np.arange(512), np.arange(512) ^ (1 << (8 - i))] = 1 / 9
flip3 = np.arange(512) ^ 511                    # 전역 반전 F: 모든 비트 XOR
ALL_UP = 511
assert is_stochastic(prop3) and E3[ALL_UP] == -18 and M3[ALL_UP] == 9
print(f"states {len(states3)}, E ∈ [{E3.min()}, {E3.max()}], 결합 수 {-E3[ALL_UP]}")

`metropolis_chain(target, proposal)`로 random-scan 사슬을 만들고 합격 기준 1을 확인한다: 행확률, 정상분포 = 볼츠만, 세부 균형.

In [ ]:
def metropolis_ising(E, prop, T):
    return metropolis_chain(np.exp(-(E - E.min()) / T), prop)

P3, pi3 = metropolis_ising(E3, prop3, T_C), boltzmann(E3, T_C)
assert is_stochastic(P3)
np.testing.assert_allclose(stationary(P3), pi3, atol=1e-12)          # 정상분포 = 볼츠만
flux = pi3[:, None] * P3
np.testing.assert_allclose(flux, flux.T, atol=1e-14)                 # 세부 균형
acc3 = float((pi3 * (1 - np.diag(P3))).sum())                        # 정상상태 수용률
exact3 = {"absm": float(pi3 @ np.abs(m3)), "e": float(pi3 @ (E3 / 9)),
          "m2": float(pi3 @ m3**2), "m": float(pi3 @ m3)}
print(f"정상성 오차 {np.abs(stationary(P3) - pi3).max():.1e}, 세부 균형 위반 {np.abs(flux - flux.T).max():.1e}")
print(f"T_c 수용률 {acc3:.6f};  exact <|m|>={exact3['absm']:.6f}, <e>={exact3['e']:.6f}, "
      f"<m^2>={exact3['m2']:.6f}, <m>={exact3['m']:.1e}")

**스펙트럼.** 가역 사슬이므로 $D^{1/2}PD^{-1/2}$($D = \mathrm{diag}\,\pi$)가 대칭 — 고유값이 실수이고 `eigh`로 안정하게 구해진다. 오른쪽 고유함수 $r_k = u_k/\sqrt\pi$는 $\ell^2(\pi)$ 정규직교이며, 전역 반전 순열 `flip3`으로 $r_k(Fx) = \pm r_k(x)$의 패리티를 읽는다.

In [ ]:
def spectrum(P, pi):
    """가역 P의 (고유값 |λ| 내림차순, 오른쪽 고유함수 r_k: ℓ²(π) 정규직교, 열벡터)."""
    sq = np.sqrt(pi)
    w, U = sla.eigh(sq[:, None] * P / sq[None, :])
    order = np.argsort(-np.abs(w))
    return w[order], U[:, order] / sq[:, None]

def parity(r, flip):
    if np.allclose(r[flip], r, atol=1e-8):
        return "even"
    return "odd" if np.allclose(r[flip], -r, atol=1e-8) else "mixed"

lam3, R3 = spectrum(P3, pi3)
lam_generic = np.sort(np.abs(np.linalg.eigvals(P3.T)))[::-1]        # 일반 eig와 교차 검증
np.testing.assert_allclose(np.abs(lam3[:6]), lam_generic[:6], atol=1e-10)
for k in range(5):
    relax = -1 / np.log(abs(lam3[k])) if k else np.inf
    print(f"λ_{k+1} = {lam3[k]:+.6f}  ({parity(R3[:, k], flip3):5s})  완화 {relax:7.1f} step = {relax/9:5.1f} sweep")
np.testing.assert_allclose(np.abs(lam3[1:3]), [0.996348, 0.944340], atol=1e-5)
assert parity(R3[:, 1], flip3) == "odd" and parity(R3[:, 2], flip3) == "even"

온도를 바꾸며 같은 계산을 반복한다. $|\lambda_2|$(홀, 터널링)는 $T$가 내려갈수록 1에 붙고, $|\lambda_3|$(짝)는 $T_c$ 근처에서 가장 크다 — 3×3에서 보이는 '임계 감속'의 씨앗이다.

In [ ]:
T_list = [1.0, 1.5, 2.0, T_C, 3.0, 5.0]
lines = ["| $T$ | $\\|\\lambda_2\\|$ (홀) | $\\|\\lambda_3\\|$ (짝) | 수용률 | $\\langle\\|m\\|\\rangle$ | $\\langle e\\rangle$ |", "|---|---|---|---|---|---|"]
for T in T_list:
    P, pi = metropolis_ising(E3, prop3, T), boltzmann(E3, T)
    np.testing.assert_allclose(stationary(P), pi, atol=1e-10)
    lam, R = spectrum(P, pi)
    assert parity(R[:, 1], flip3) == "odd" and parity(R[:, 2], flip3) == "even"
    lines.append(f"| {T:.4f} | {abs(lam[1]):.6f} | {abs(lam[2]):.5f} | {(pi * (1 - np.diag(P))).sum():.4f} "
                 f"| {pi @ np.abs(m3):.5f} | {pi @ (E3 / 9):.5f} |")
Markdown("\n".join(lines))

**TV 곡선(합격 기준 2).** 행벡터 $\mu P^n$을 직접 곱해 $\mathrm{TV}(\mu P^n, \pi)$를 계산한다. all-up($F$-비대칭 점질량)과 균등($F$-대칭) 시작을 비교하고, 참조선 $|\lambda_2|^n$, $|\lambda_3|^n$을 겹친다. 같은 루프에서 홀 관측량 $E_{\text{all-up}}[m(X_n)]$도 저장해 둔다(Task 2, E2에서 사용).

In [ ]:
n_max = 500
row_up, row_uni = np.zeros(512), np.full(512, 1 / 512)
row_up[ALL_UP] = 1.0
tv_up, tv_uni, Em_up = (np.empty(n_max + 1) for _ in range(3))
for n in range(n_max + 1):
    tv_up[n], tv_uni[n], Em_up[n] = tv_distance(row_up, pi3), tv_distance(row_uni, pi3), row_up @ m3
    row_up, row_uni = row_up @ P3, row_uni @ P3
n_show = [1, 5, 10, 20, 50, 100, 200, 300, 500]
print("n      TV(all-up)  TV(uniform)  E_allup[m(X_n)]")
for n in n_show:
    print(f"{n:4d}   {tv_up[n]:.4f}      {tv_uni[n]:.3e}    {Em_up[n]:.5f}")
assert tv_uni[300] < 1e-6 and tv_up[300] > 0.1

ns = np.arange(n_max + 1)
fig, ax = plt.subplots()
ax.semilogy(ns, tv_up, label="from all-up (point mass, F-asymmetric)")
ax.semilogy(ns, tv_uni, label="from uniform (F-symmetric)")
ax.semilogy(ns, abs(lam3[1]) ** ns, "k--", lw=0.8, label=r"$|\lambda_2|^n$ (odd mode)")
ax.semilogy(ns, abs(lam3[2]) ** ns, "k:", lw=0.8, label=r"$|\lambda_3|^n$ (even mode)")
ax.set(xlabel="step n", ylabel="TV distance to Boltzmann", ylim=(1e-14, 1),
       title=r"3x3 Ising Metropolis chain at $T_c$: TV distance vs start")
ax.legend(loc="lower left");

두 곡선의 기울기가 다른 것이 정리 3이다: 균등 시작은 짝 모드만 남아 $|\lambda_3|^n$으로 떨어지고(300 step에 $10^{-8}$), all-up 시작은 홀 모드 $|\lambda_2|^n$에 묶여 500 step 뒤에도 $0.08$이다.

### Task 2 — $3\times3$ 사슬을 `simulate_chain`으로 표본화해 정확값과 대조 (00b 재사용)

균등 초기분포(= $F$-대칭)로 $n=200$ step 돌리면 TV $\approx10^{-5}$이므로 마지막 상태는 사실상 볼츠만 표본이고, 경로들이 독립이라 `mc_estimate`의 iid SE가 정당하다(합격 기준 3).

In [ ]:
n_paths3 = scale(10_000, minimum=1000)
X_uni = simulate_chain(P3, np.full(512, 1 / 512), 200, rng, n_paths=n_paths3)
fin = X_uni[:, -1]
est3 = {"absm": mc_estimate(np.abs(m3[fin])), "e": mc_estimate(E3[fin] / 9),
        "m2": mc_estimate(m3[fin] ** 2), "m": mc_estimate(m3[fin])}
for k, label in [("absm", "<|m|>"), ("e", "<e>  "), ("m2", "<m^2>"), ("m", "<m>  ")]:
    print(f"{label} MC {est3[k]}   exact {exact3[k]:+.6f}   z = {(est3[k].mean - exact3[k]) / est3[k].se:+.2f}")

support = np.arange(-9, 10, 2)                                   # M ∈ {-9, -7, ..., 9}
pmf_M = np.array([pi3[M3 == k].sum() for k in support])
ax = plot_hist_vs_pmf(M3[fin], support, pmf_M, label_pmf="exact Boltzmann pmf")
ax.set(xlabel="M = sum of spins  (m = M/9)", ylabel="probability",
       title=f"3x3 Ising at $T_c$: M after 200 steps, uniform start ({n_paths3} chains)");

**부가: 홀 관측량의 느린 감쇠.** all-up 점질량에서 시작하면 $E[m(X_n)]$은 정확히 $e_{\text{up}}P^n m$이고 $\lambda_2^n$으로 감쇠한다(정리 3, E2). `simulate_chain`으로 $n = 100, 200, 300$을 추정해 Task 1에서 계산한 정확값과 대조한다.

In [ ]:
X_up = simulate_chain(P3, ALL_UP, 300, rng, n_paths=scale(10_000))
est_up = {n: mc_estimate(m3[X_up[:, n]]) for n in (100, 200, 300)}
for n in (100, 200, 300):
    print(f"E_allup[m(X_{n})]  MC {est_up[n]}   exact {Em_up[n]:.5f}   z = {(est_up[n].mean - Em_up[n]) / est_up[n].se:+.2f}")
print(f"exact ratio E[m_300]/E[m_200] = {Em_up[300] / Em_up[200]:.4f}  vs  |λ2|^100 = {abs(lam3[1]) ** 100:.4f}")

### Task 3 — 체커보드 메트로폴리스 샘플러 구현과 $4\times4$ 완전 열거 검증

$L$이 커지면 $2^{L^2}$ 상태의 행렬은 불가능하고 random-scan 파이썬 루프도 느리다. 정리 2의 체커보드 갱신은 같은 색 사이트 $N/2$개를 numpy 한 줄로 동시에 처리한다. 배치 축 $B$에는 온도(Task 4) 또는 독립 복제(Task 3)를 싣는다.

In [ ]:
# helper (spkit에 없어서 여기서 정의): 체커보드 메트로폴리스 샘플러
def ising_checkerboard_sweep(s, beta, rng, mask_even):
    """in-place 1 sweep: 짝 부분격자 전체 → 홀 부분격자 전체. s: (B, L, L) int8, beta: (B, 1, 1)."""
    for mask in (mask_even, ~mask_even):
        nb = np.roll(s, 1, 1) + np.roll(s, -1, 1) + np.roll(s, 1, 2) + np.roll(s, -1, 2)
        dE = 2 * s * nb                                   # ΔE ∈ {-8, -4, 0, 4, 8}
        accept = (dE <= 0) | (rng.random(s.shape) < np.exp(-beta * dE))
        s[accept & mask] *= -1

def ising_run(L, temps, n_sweeps, burn, rng, R=1, s0="up"):
    """temps 각각을 R개 복제로 동시에 돌린다. 반환 (m_series, e_series): shape (n_sweeps - burn, len(temps)*R),
    열 순서는 [T0 복제 0..R-1, T1 복제 0..R-1, ...]."""
    temps = np.atleast_1d(np.asarray(temps, dtype=float))
    B, N = temps.size * R, L * L
    beta = np.repeat(1 / temps, R).reshape(B, 1, 1)
    s = (np.ones((B, L, L), dtype=np.int8) if s0 == "up"
         else rng.choice(np.array([-1, 1], dtype=np.int8), size=(B, L, L)))
    ii, jj = np.indices((L, L))
    mask_even = (((ii + jj) % 2) == 0)[None]
    m_ser, e_ser = np.empty((n_sweeps - burn, B)), np.empty((n_sweeps - burn, B))
    for t in range(n_sweeps):
        ising_checkerboard_sweep(s, beta, rng, mask_even)
        if t >= burn:
            m_ser[t - burn], e_ser[t - burn] = s.sum(axis=(1, 2)) / N, ising_energy(s) / N
    return m_ser, e_ser

$4\times4$의 65,536개 상태는 비트 트릭으로 한 번에 열거된다. $R=200$개 독립 복제를 all-up에서 시작해 각 복제의 시간평균을 하나의 표본으로 삼으면(정리 5) SE가 자기상관에 무관하게 정당하다(합격 기준 4).

In [ ]:
bits = ((np.arange(2**16)[:, None] >> np.arange(16)) & 1).astype(np.int8) * 2 - 1
states4 = bits.reshape(-1, 4, 4)
E4, M4 = ising_energy(states4), states4.sum(axis=(1, 2))

def exact4(T):
    pi, m = boltzmann(E4, T), M4 / 16
    return {"absm": float(pi @ np.abs(m)), "e": float(pi @ (E4 / 16)),
            "m2": float(pi @ m**2), "m4": float(pi @ m**4)}

np.testing.assert_allclose(exact4(T_C)["absm"], 0.843860, atol=1e-5)   # 열거 코드 자체 점검 (참고값 0.8438604)
R, T4 = 200, [T_C, 2.5]
n_sw4, burn4 = scale(5000, minimum=500), scale(500, minimum=100)
t0 = time.perf_counter()
m4s, e4s = ising_run(4, T4, n_sw4, burn4, rng, R=R)
print(f"4x4: R={R}, {n_sw4} sweeps (burn {burn4}) × {len(T4)} temps: {time.perf_counter() - t0:.2f} s")
rep = {}
for k, T in enumerate(T4):
    blk = slice(k * R, (k + 1) * R)
    rep[T] = {"absm": mc_estimate(np.abs(m4s[:, blk]).mean(0)), "e": mc_estimate(e4s[:, blk].mean(0))}
    ex = exact4(T)
    print(f"T={T:.4f}  <|m|> MC {rep[T]['absm']}  exact {ex['absm']:.6f} (z={(rep[T]['absm'].mean - ex['absm']) / rep[T]['absm'].se:+.2f})"
          f" | <e> MC {rep[T]['e']}  exact {ex['e']:.6f} (z={(rep[T]['e'].mean - ex['e']) / rep[T]['e'].se:+.2f})")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, key, lab in zip(axes, ["absm", "e"], [r"$\langle|m|\rangle$", r"$\langle e\rangle$"]):
    ax.errorbar(T4, [rep[T][key].mean for T in T4], yerr=[1.96 * rep[T][key].se for T in T4],
                fmt="o", capsize=4, label="checkerboard MC (95% CI)")
    ax.plot(T4, [exact4(T)[key] for T in T4], "x", ms=10, mew=2, color="C3", label="exact enumeration")
    ax.set(xlabel="T", ylabel=lab, xticks=T4, xticklabels=[f"{T:.3f}" for T in T4])
    ax.legend()
fig.suptitle(f"4x4 Ising: checkerboard Metropolis vs exact (R = {R} replicas)");

### Task 4 — 자화 곡선 $\langle|m|\rangle(T)$, $\langle e\rangle(T)$, $\chi(T)$, $C(T)$ vs Onsager

온도 배열을 배치 축에 실어 모든 $T$를 동시에 갱신한다. 시작은 all-up(저온 상에서 출발) — 고온에서는 어차피 빨리 잊히고, 저온에서는 두 상 사이의 터널링 문제를 피한다. 오차막대는 Task 5의 $\tau_{int}$로 보정한 SE $= \mathrm{sd}\sqrt{2\tau_{int}/n}$이므로 먼저 헬퍼를 정의한다.

In [ ]:
# helper (spkit에 없어서 여기서 정의): 자기상관 시간, Onsager, 열역학 관측량
def acf(x, n_lags):
    """정규화 자기상관 ρ_0..ρ_{n_lags-1} (FFT, 편향 없는 정규화)."""
    x = np.asarray(x, dtype=float) - np.mean(x)
    n = x.size
    f = np.fft.rfft(x, 2 * n)
    acov = np.fft.irfft(f * np.conj(f))[:n] / np.arange(n, 0, -1)
    return acov[:n_lags] / acov[0] if acov[0] > 0 else np.r_[1.0, np.zeros(n_lags - 1)]

def autocorr_time(x, c=6):
    """τ_int = 1/2 + Σ_{k≤M} ρ_k, Sokal 자동 창: 처음으로 M ≥ c·τ 가 되는 M에서 절단."""
    rho, tau = acf(x, len(x)), 0.5
    for M in range(1, len(rho)):
        tau += rho[M]
        if M >= c * tau:
            break
    return max(tau, 0.5)

def onsager_m(T):
    T = np.asarray(T, dtype=float)
    with np.errstate(invalid="ignore"):
        return np.where(T < T_C, (1 - np.sinh(2 / T) ** -4.0) ** (1 / 8), 0.0)

def chi(m_series, T, N):            # 자화율: N β (<m²> - <|m|>²)
    return N / T * (np.mean(m_series**2, 0) - np.mean(np.abs(m_series), 0) ** 2)

def specific_heat(e_series, T, N):  # 비열: N β² (<e²> - <e>²)
    return N / T**2 * np.var(e_series, 0)

def binder(m_series):
    return 1 - np.mean(m_series**4, 0) / (3 * np.mean(m_series**2, 0) ** 2)

np.testing.assert_allclose(T_C, 2.269185314213022, rtol=1e-12)
np.testing.assert_allclose(onsager_m(2.0), 0.911319, atol=1e-6)
print(f"onsager_m: {[f'{onsager_m(T):.5f}' for T in (1.5, 2.0, 2.2, 2.25, 2.3)]}")

In [ ]:
if fast():
    L_MAIN, temps = 16, np.array([1.5, 1.8, 2.0, 2.1, 2.2, T_C, 2.35, 2.5, 3.0])
    n_sw, burn = 4000, 1000
else:
    L_MAIN, temps = 32, np.array([1.5, 1.8, 2.0, 2.1, 2.15, 2.2, 2.25, T_C, 2.3, 2.35, 2.5, 2.75, 3.0])
    n_sw, burn = 20_000, 4000
N_MAIN, n_samp = L_MAIN**2, n_sw - burn
t0 = time.perf_counter()
m_main, e_main = ising_run(L_MAIN, temps, n_sw, burn, rng)
print(f"L={L_MAIN}: {len(temps)} temps × {n_sw} sweeps (burn {burn}) in {time.perf_counter() - t0:.1f} s")

absm_main = np.abs(m_main)
tau_main = np.array([autocorr_time(absm_main[:, k]) for k in range(len(temps))])
tau_e = np.array([autocorr_time(e_main[:, k]) for k in range(len(temps))])
obs = {"absm": absm_main.mean(0), "e": e_main.mean(0), "chi": chi(m_main, temps, N_MAIN),
       "C": specific_heat(e_main, temps, N_MAIN), "U": binder(m_main)}
se_absm = absm_main.std(0, ddof=1) * np.sqrt(2 * tau_main / n_samp)     # τ 보정 SE (정리 5)
se_e = e_main.std(0, ddof=1) * np.sqrt(2 * tau_e / n_samp)
lines = [f"| $T$ | $\\langle\\|m\\|\\rangle$ ± SE | Onsager $m(T)$ | $\\langle e\\rangle$ ± SE | $\\chi$ | $C$ | $U$ | $\\tau_{{int}}(\\|m\\|)$ |", "|---|---|---|---|---|---|---|---|"]
for k, T in enumerate(temps):
    lines.append(f"| {T:.4f} | {obs['absm'][k]:.4f} ± {se_absm[k]:.4f} | {onsager_m(T):.4f} | {obs['e'][k]:.4f} ± {se_e[k]:.4f} "
                 f"| {obs['chi'][k]:.2f} | {obs['C'][k]:.2f} | {obs['U'][k]:.3f} | {tau_main[k]:.1f} |")
Markdown("\n".join(lines))

In [ ]:
T_fine = np.linspace(1.2, 3.2, 400)
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
ax = axes[0, 0]
ax.errorbar(temps, obs["absm"], yerr=1.96 * se_absm, fmt="o-", capsize=3, label=f"MC L={L_MAIN} (95% CI)")
ax.plot(T_fine, onsager_m(T_fine), "C3-", lw=1.2, label=r"Onsager $m(T)$, $L\to\infty$")
ax.set(ylabel=r"$\langle|m|\rangle$", ylim=(0, 1.05))
axes[0, 1].errorbar(temps, obs["e"], yerr=1.96 * se_e, fmt="o-", capsize=3, label=f"MC L={L_MAIN}")
axes[0, 1].set(ylabel=r"$\langle e\rangle$ (energy per spin)")
axes[1, 0].plot(temps, obs["chi"], "o-"); axes[1, 0].set(ylabel=r"susceptibility $\chi$")
axes[1, 1].plot(temps, obs["C"], "o-"); axes[1, 1].set(ylabel=r"specific heat $C$")
for ax in axes.ravel():
    ax.axvline(T_C, color="gray", ls="--", lw=0.8, label=r"$T_c = 2.269$")
    ax.set(xlabel="T")
    ax.legend()
fig.suptitle(f"2D Ising L={L_MAIN}: magnetization curve vs Onsager, energy, chi, C")
fig.tight_layout()
k_Tc, k_T2 = int(np.argmin(np.abs(temps - T_C))), int(np.argmin(np.abs(temps - 2.0)))
print(f"<|m|>(T_c) = {obs['absm'][k_Tc]:.3f} (L={L_MAIN});  T=2.0: {obs['absm'][k_T2]:.4f} vs Onsager {onsager_m(2.0):.4f}")
print(f"chi 최대 T = {temps[np.argmax(obs['chi'])]:.3f},  C 최대 T = {temps[np.argmax(obs['C'])]:.3f}")

$T<2.1$에서는 점들이 Onsager 곡선 위에 얹히고, $T_c$ 근처에서 급락하되 0이 되지 않는다(유한 크기). $\chi$의 봉우리는 $T_c$보다 높은 쪽($L=32$: $\approx2.35$; $L=16$은 봉우리가 넓어 $2.35$–$2.5$)에 있다 — 유한 크기 이동(finite-size shift)이다. $C$의 봉우리는 $T_c$에 더 가깝다.

### Task 5 — 자기상관 시간과 임계 감속 (05a/00b 재사용)

Task 4의 $|m|$ 시계열은 $L_{\text{MAIN}}$의 $\tau_{int}(T)$를 이미 주었다. 다른 $L$을 같은 온도 격자에서 돌려 $\tau_{int}$가 $T_c$ 근방에서 봉우리를 이루고 $L$과 함께 커지는지 본다(합격 기준 5 뒷부분).

In [ ]:
L_extra = [8] if fast() else [8, 16]
absm_by_L = {L_MAIN: absm_main}
t0 = time.perf_counter()
for L in L_extra:
    mL, _ = ising_run(L, temps, n_sw, burn, rng)
    absm_by_L[L] = np.abs(mL)
print(f"extra lattices {L_extra}: {time.perf_counter() - t0:.1f} s")
tau_by_L = {L: np.array([autocorr_time(absm_by_L[L][:, k]) for k in range(len(temps))]) for L in sorted(absm_by_L)}
absm_by_L_mean = {L: absm_by_L[L].mean(0) for L in tau_by_L}
k_15 = int(np.argmin(np.abs(temps - 1.5)))
print("L    tau(T_c)  tau(T=1.5)  <|m|>(T_c)")
for L in tau_by_L:
    print(f"{L:2d}   {tau_by_L[L][k_Tc]:6.2f}    {tau_by_L[L][k_15]:6.2f}      {absm_by_L_mean[L][k_Tc]:.3f}")
assert tau_by_L[16][k_Tc] > tau_by_L[16][k_15], "critical slowing down: tau(T_c) should exceed tau(T=1.5)"

fig, ax = plt.subplots(figsize=(8, 4.5))
for L in tau_by_L:
    ax.semilogy(temps, tau_by_L[L], "o-", label=f"L = {L}")
ax.axvline(T_C, color="gray", ls="--", lw=0.8, label=r"$T_c$")
ax.set(xlabel="T", ylabel=r"$\tau_{int}(|m|)$ [sweeps]", title=r"Integrated autocorrelation time of $|m|$: critical slowing down")
ax.legend(loc="upper left")
ins = ax.inset_axes([0.62, 0.55, 0.35, 0.4])
for k, lab in [(k_Tc, r"$T_c$"), (k_15, "T = 1.5")]:
    ins.plot(acf(absm_by_L[16][:, k], 60), lw=1.2, label=lab)
ins.axhline(0, color="k", lw=0.5)
ins.set(title="ACF of |m|, L = 16", xlabel="lag [sweeps]", ylim=(-0.1, 1))
ins.legend(fontsize=8);

**유효 표본 수와 보정 SE.** $n$ sweep의 시계열을 iid로 착각하면 SE가 $\sqrt{2\tau_{int}}$배 작게 나온다(함정 4). $L_{\text{MAIN}}$에서 온도별로 비교한다.

In [ ]:
lines = ["| $T$ | $\\tau_{int}(\\|m\\|)$ | $n_{eff} = n/2\\tau$ | naive SE $= \\mathrm{sd}/\\sqrt n$ | 보정 SE $= \\mathrm{sd}\\sqrt{2\\tau/n}$ | 과소평가 배율 |", "|---|---|---|---|---|---|"]
sd = absm_main.std(0, ddof=1)
for k, T in enumerate(temps):
    lines.append(f"| {T:.3f} | {tau_main[k]:.2f} | {n_samp / (2 * tau_main[k]):.0f} | {sd[k] / np.sqrt(n_samp):.5f} "
                 f"| {se_absm[k]:.5f} | {np.sqrt(2 * tau_main[k]):.2f}× |")
Markdown("\n".join(lines))

봉우리가 $T_c$–$2.35$에 있고 $L$이 커질수록 높아진다: 임계 감속이다. 이론은 $\tau\sim L^{2.17}$이지만 여기서는 실행이 짧아(Sokal 창 노이즈) 지수가 대략 1–2로 나온다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

정확값은 모두 위에서 열거·행렬로 계산한 값이다(하드코딩 없음). 마지막 행의 Onsager 값은 $L\to\infty$ 극한이라 '닫힌 형식' 칸을 비워 두고 참고로만 적는다.

In [ ]:
p = predictions
rows = [
    {"name": "3x3 at T_c: <|m|> (simulate_chain, uniform start, n=200)", "predicted": p["absm_3x3_Tc"], "estimate": est3["absm"], "exact": exact3["absm"]},
    {"name": "3x3 at T_c: <e>", "predicted": None, "estimate": est3["e"], "exact": exact3["e"]},
    {"name": "3x3 at T_c: <m^2>", "predicted": None, "estimate": est3["m2"], "exact": exact3["m2"]},
    {"name": "3x3 at T_c: <m> (signed; symmetry)", "predicted": None, "estimate": est3["m"], "exact": exact3["m"]},
    {"name": "3x3 from all-up: E[m(X_200)] (odd observable, slow decay)", "predicted": None, "estimate": est_up[200], "exact": float(Em_up[200])},
    {"name": "4x4 at T_c: <|m|> (checkerboard, R=200 replicas)", "predicted": None, "estimate": rep[T_C]["absm"], "exact": exact4(T_C)["absm"]},
    {"name": "4x4 at T_c: <e>", "predicted": None, "estimate": rep[T_C]["e"], "exact": exact4(T_C)["e"]},
    {"name": "4x4 at T = 2.5: <|m|>", "predicted": None, "estimate": rep[2.5]["absm"], "exact": exact4(2.5)["absm"]},
    {"name": "4x4 at T = 2.5: <e>", "predicted": None, "estimate": rep[2.5]["e"], "exact": exact4(2.5)["e"]},
    {"name": f"L={L_MAIN} at T = 2.0: <|m|> (tau-corrected SE; Onsager {onsager_m(2.0):.4f} is L→∞, 참고)",
     "predicted": p["m_onsager_T2"], "estimate": float(obs["absm"][k_T2]), "se": float(se_absm[k_T2]), "exact": None},
]
Markdown(compare_table(rows))

- 행 1–4가 어긋나면: `simulate_chain`의 초기분포·step 수 문제이거나 `exact3`의 열거(결합 중복·부호)가 틀린 것이다. $\langle m\rangle\ne0$이면 $F$-대칭이 깨진 것.
- 행 5가 어긋나면: 홀 모드 감쇠 $\lambda_2^n$의 정확값 `Em_up`(행벡터 × $P^n$)과 경로의 시작 상태가 맞지 않는 것 — 특히 all-up 인덱스.
- 행 6–9가 어긋나면: 체커보드 마스크(같은 색이 이웃이 되는 실수), $\Delta E$의 부호·계수 2, 또는 burn-in 부족. 정리 2의 가정 위반이 가장 흔한 원인.
- 행 10은 판정 대상이 아니다. $L=32$면 Onsager와 $0.002$ 이내, $L=16$이면 $0.001$ 정도 위에 있어야 하며 크게 벗어나면 워밍업이 부족한 것.

In [ ]:
assert_close(est3["absm"], exact3["absm"], k=4, name="3x3 <|m|>")
assert_close(est3["e"], exact3["e"], k=4, name="3x3 <e>")
assert_close(est3["m2"], exact3["m2"], k=4, name="3x3 <m^2>")
assert_close(est3["m"], exact3["m"], k=4, name="3x3 <m> signed")
assert_close(est_up[200], float(Em_up[200]), k=4, name="E_allup[m(X_200)]")
for T in T4:
    assert_close(rep[T]["absm"], exact4(T)["absm"], k=4, name=f"4x4 <|m|> T={T:.3f}")
    assert_close(rep[T]["e"], exact4(T)["e"], k=4, name=f"4x4 <e> T={T:.3f}")
assert abs(acc3 - 0.136989) < 1e-6
print("모든 assert_close 통과 (k=4)")

## 6. 연습문제

### E1 계산

$3\times3$ 토러스에서 all-up 상태의 에너지 $E$와 스핀당 $e$, 스핀 하나를 뒤집었을 때 $\Delta E$, $T_c$에서 그 뒤집기의 수용확률, 그리고 random-scan 사슬이 all-up을 한 step에 떠날 확률을 손으로 구하라. 왜 $3\times3$의 $\langle|m|\rangle$가 $T_c$에서 $0.87$이나 되는지 설명하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

결합 수 $2N = 18$, 모두 정렬 → $E = -18$, $e = -2$. 한 스핀 뒤집기: 이웃 4개와의 결합 4개가 $-1\to+1$ → $\Delta E = 2\cdot1\cdot4 = 8$. 수용확률 $e^{-8/T_c} = e^{-3.5255} = 0.029437$. 어느 사이트를 제안하든 같으므로 떠날 확률 $= 9\times\frac19\times0.029437 = 0.029437$, 즉 $P(\text{all-up}, \text{all-up}) = 0.970563$.

$N=9$에서는 $|m|\in\{1/9, 3/9, \dots, 1\}$로 0이 될 수 없고, 정렬 상태 두 개의 볼츠만 가중치 $e^{18/T_c}$가 압도적이라($T_c$에서 이 두 상태만으로 전체 확률의 약 67%를 차지) 평균이 크다 — 유한 크기 효과의 극단이다.

```python
dE = 8
print(E3[ALL_UP], E3[ALL_UP] / 9, np.exp(-dE / T_C), 1 - P3[ALL_UP, ALL_UP])   # -18, -2.0, 0.029437, 0.029437
print("P(all-up or all-down) =", pi3[ALL_UP] + pi3[0])                            # ≈ 0.674
```

</details>

### E2 유도 후 시뮬레이션 검증

정리 3을 써서 all-up에서 시작한 $3\times3$ 사슬의 홀 관측량 $E[m(X_n)]$이 $n\to\infty$에서 $\approx c\,\lambda_2^n$으로 감쇠함을 유도하라. `simulate_chain(P3, x0=ALL_UP, n_steps=300, rng, n_paths=scale(10000))`로 $n=100,200,300$의 $E[m(X_n)]$을 추정해 정확값(행벡터 × $P^n$, Task 1의 `Em_up`)과 `assert_close`하고, 비율 $E[m_{300}]/E[m_{200}]$이 $\lambda_2^{100} = 0.6936$에 가까운지 보라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

가역 사슬의 스펙트럼 전개(정리 3 스케치): $E_x[m(X_n)] = (P^n m)(x) = \sum_k\lambda_k^n r_k(x)\langle m, r_k\rangle_\pi$. $m$은 홀($m\circ F = -m$)이므로 짝 $r_k$와의 내적 $\langle m, r_k\rangle_\pi = \sum_x\pi(x)m(x)r_k(x)$는 $x\to Fx$ 치환에서 부호가 바뀌어 0이고, $\lambda_1 = 1$ 항도 $r_1\equiv1$이 짝이라 사라진다($\pi(m) = 0$). 남은 홀 모드 중 가장 큰 $\lambda_2 = 0.996348$이 지배하므로 $E[m(X_n)] \approx c\lambda_2^n$, $c = r_2(\text{all-up})\langle m, r_2\rangle_\pi$. 따라서 $E[m_{300}]/E[m_{200}] \to \lambda_2^{100} = 0.6936$. 정확값: $n=50$: $0.80849$, $100$: $0.67324$, $200$: $0.46697$, $300$: $0.32390$. SE $\approx 0.9/\sqrt{n_{paths}}$ (sd$(m)\approx0.90$).

```python
coef = np.array([pi3 @ (m3 * R3[:, k]) for k in range(512)])          # <m, r_k>_π
print("짝 모드와의 내적 최대:", max(abs(coef[k]) for k in range(512) if parity(R3[:, k], flip3) == "even"))  # ~1e-14
n = np.array([50, 100, 200, 300])
spectral = [(lam3**k * R3[ALL_UP] * coef).sum() for k in n]              # 전개로 재계산 = Em_up[n]
print(np.allclose(spectral, Em_up[n]), Em_up[n])
X = simulate_chain(P3, ALL_UP, 300, rng, n_paths=scale(10_000))
for k in (100, 200, 300):
    est = mc_estimate(m3[X[:, k]])
    assert_close(est, float(Em_up[k]), k=4, name=f"E_allup[m(X_{k})]")
    print(k, est, f"exact {Em_up[k]:.5f}")
print("ratio exact", Em_up[300] / Em_up[200], "λ2^100 =", abs(lam3[1]) ** 100)
```

</details>

### E3 가정을 깨보기

(a) 체커보드 대신 모든 사이트를 동시에 메트로폴리스 갱신하는 '전체 병렬' 샘플러를 $4\times4$에 적용해 $T=2.0$과 $T_c$에서 $\langle|m|\rangle$, $\langle e\rangle$를 추정하고 정확값과 비교하라. 무엇이 깨졌는가? (b) $\beta=0$($T=\infty$)에서 체커보드 샘플러를 돌리면 $|m|$이 어떻게 되는가? random-scan $3\times3$ 사슬은 $\beta=0$에서 어떤 사슬이 되며 왜 점질량 초기분포에서 수렴하지 않는가(01e)?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 참고 실행($R=200$, 3,000 sweep): $T=2.0$: $\langle|m|\rangle = 0.000$, $\langle e\rangle = +1.93$ (정확 $0.919$, $-1.755$); $T_c$: $0.001$, $+1.87$ (정확 $0.844$, $-1.566$). 이웃을 동시에 뒤집으면 각 사이트가 '옛' 이웃을 보고 판단한다: 정렬 상태에서는 아무도 뒤집지 않으려 하지만 $\Delta E\le0$인 사이트들이 **함께** 뒤집히면서 서로의 전제가 무너진다. 정리 2의 가환성(같은 색끼리만 동시 갱신)이 깨져 곱 커널 구조가 사라지고 정상분포가 볼츠만이 아니다 — 사슬은 반강자성 체커보드 상태($e>0$, 모든 이웃이 반대 부호)에 갇힌다. 그 상태에서는 모든 사이트가 $\Delta E = -8$이라 전부 동시에 뒤집히고, 다시 반강자성 상태가 된다(주기 2).

(b) $\beta=0$이면 모든 제안이 수용된다: 짝 부분격자 전부 뒤집고 이어서 홀 부분격자 전부 뒤집음 = 결정론적 전역 반전 $s\mapsto -s$. $|m|$은 초기값 그대로다(1, 0.375, 0.5 등) — 에르고딕성이 완전히 사라진다. random-scan $3\times3$은 제안행렬 그대로, 즉 9차원 하이퍼큐브 위 단순 랜덤 워크가 된다: 이분 그래프라 주기 2, 정상분포(균등)는 있으나 점질량 초기분포에서 $P^n$이 진동한다 — 01e의 주기적 반례. (전역 반전에서 $|\lambda_2|$가 1에 붙는 저온과 달리, 여기서는 $\lambda = -1$이 생긴다.)

```python
def ising_parallel_sweep(s, beta, rng):                  # 틀린 샘플러: 모든 사이트 동시 갱신
    nb = np.roll(s, 1, 1) + np.roll(s, -1, 1) + np.roll(s, 1, 2) + np.roll(s, -1, 2)
    dE = 2 * s * nb
    s[(dE <= 0) | (rng.random(s.shape) < np.exp(-beta * dE))] *= -1

for T in (2.0, T_C):
    s, beta = np.ones((200, 4, 4), dtype=np.int8), np.full((200, 1, 1), 1 / T)
    ms, es = [], []
    for t in range(3000):
        ising_parallel_sweep(s, beta, rng)
        if t >= 500:
            ms.append(np.abs(s.sum(axis=(1, 2))) / 16); es.append(ising_energy(s) / 16)
    print(f"parallel T={T:.3f}: <|m|>={np.mean(ms):.3f}, <e>={np.mean(es):+.3f}   exact {exact4(T)['absm']:.3f}, {exact4(T)['e']:.3f}")

s = rng.choice(np.array([-1, 1], dtype=np.int8), size=(3, 4, 4)); m_before = s.sum(axis=(1, 2)) / 16
ii, jj = np.indices((4, 4)); mask_even = (((ii + jj) % 2) == 0)[None]
for t in range(10):
    ising_checkerboard_sweep(s, np.zeros((3, 1, 1)), rng, mask_even)
print("beta=0: m before", m_before, "after 10 sweeps", s.sum(axis=(1, 2)) / 16)   # 부호까지 같음 (짝수 번 반전)
P0 = metropolis_ising(E3, prop3, np.inf)                 # β=0: P = 제안행렬
print(np.allclose(P0, prop3), np.sort(np.abs(np.linalg.eigvals(P0)))[-2:])   # True, [1, 1] → λ = -1 (주기 2)
```

</details>

### 스트레치 S1–S3 (선택)

**S1 (Binder 누적률).** $U_L(T) = 1 - \langle m^4\rangle/(3\langle m^2\rangle^2)$를 $L=8,16,32$에서 계산해 곡선들의 교차점으로 $T_c$를 추정하라. 교차점의 값과 보편값 $U^*\approx0.6107$을 비교하라.

**S2 (Wolff 군집 알고리즘).** 결합 확률 $p = 1-e^{-2\beta J}$로 같은 스핀 이웃을 BFS로 묶어 군집 전체를 뒤집는 Wolff 갱신을 inline으로 구현하고, $4\times4$ 정확값으로 검증한 뒤 $L=32$, $T_c$에서 $\tau_{int}(|m|)$이 메트로폴리스(≈40 sweep)보다 얼마나 작은지 재라.

**S3 (유한 크기 스케일링).** $\chi_{max}(L) \sim L^{\gamma/\nu}$, $\gamma/\nu = 7/4$. Task 5의 $L=8,16,32$ 자료에서 $\chi$의 최댓값 비율을 구해 $2^{7/4} = 3.364$와 비교하고, $T_c$에서 $\langle|m|\rangle_L\sim L^{-\beta/\nu}$ ($\beta/\nu = 1/8$)도 확인하라.

<details><summary>정답(요약)</summary>

**S1.** `ising_run`이 $m$ 시계열을 주므로 `binder(m_series)`를 $L$별로 계산하면 된다. 20,000 sweep 참고값 $U_L(T_c)$: $L=8$: $0.609$, $16$: $0.617$, $32$: $0.604$ — 모두 $0.61$ 근처에서 교차(노이즈 $\pm0.01$; $4\times4$ 정확값은 $0.6172$). $T=2.5$에서는 $0.531/0.392/0.143$로 갈라진다. 교차점 온도 추정 $\approx2.27\pm0.02$.

**S2.** 군집 갱신은 Fortuin–Kasteleyn 표현에서 세부 균형을 만족한다(Newman & Barkema Ch. 4). $T_c$에서 $\tau$는 군집 갱신 몇 회 수준($\approx1$–$3$)으로 임계 감속이 거의 사라진다($z\approx0.25$). 파이썬 BFS라 갱신당 느리지만 $\tau$가 작아 총 비용은 비슷하거나 낫다.

**S3.** 참고값 $\chi_{max}$: $1.7$ ($L=8$), $5.5$ ($16$), $21.0$ ($32$) → 비율 $3.25$, $3.79$ (평균 $\approx3.5$, 이론 $3.36$). $\langle|m|\rangle_L(T_c)$: $0.771$, $0.725$, $0.640$ → 비율 $0.94$, $0.88$ vs $2^{-1/8} = 0.917$. 온도 격자가 성기고 실행이 짧아 $\pm10\%$ 정도는 정상이다.

</details>

## 7. 정리

1. 이징 볼츠만 분포를 목표로 하는 메트로폴리스 사슬은 세부 균형으로 정상분포가 보장된다; $3\times3$(512 상태)에서 `metropolis_chain`으로 만든 $P$가 이를 $10^{-14}$까지 만족했고 수용률은 $0.137$이다.
2. 체커보드 병렬 갱신은 같은 색 사이트가 비인접이라 가환 가역 커널의 곱이 되어 옳다(정리 2); $4\times4$ 완전 열거($T_c$: $\langle|m|\rangle = 0.8439$, $\langle e\rangle = -1.5656$)와 4 SE 이내로 일치했다. '전부 동시에'는 반강자성 쓰레기를 준다(E3).
3. 반전 대칭이 고유모드를 홀/짝으로 가른다: all-up에서는 TV가 $\lambda_2^n$($0.9963$)으로 느리게 줄지만 균등 시작이나 짝 관측량($|m|, e$)은 $\lambda_3^n$($0.944$)으로 빠르게 수렴한다 — TV가 크다고 관측량이 틀린 것은 아니다.
4. $L=16/32$ 자화 곡선은 $T<2.1$에서 Onsager $m(T)$와 겹치고 $T_c = 2.269$ 근처에서 급락하되 유한 크기 때문에 0이 되지 않는다($L=16$: $0.72$, $L=32$: $0.64$).
5. $\tau_{int}(|m|)$은 $T_c$ 근방에서 봉우리를 이루고 $L$과 함께 커진다(8: ≈3, 16: ≈9–12, 32: ≈40 sweep) — 임계 감속; 올바른 SE는 독립 복제나 $2\tau_{int}$ 보정에서 나온다(05a).

**trap 카드.** Q: $L=16$에서 $T_c$의 $\langle|m|\rangle$가 $0.72$인데 Onsager는 0이라고 한다. 샘플러가 틀렸나? → A: 아니다. 유한 격자에서 $|m|>0$은 항상이고 상전이는 $L\to\infty$에서만 날카롭다: $L=8$: $0.77$, $16$: $0.72$, $32$: $0.64$로 $L^{-1/8}$을 따라 줄어든다. 샘플러 검증은 같은 크기의 정확값($4\times4$ 열거)으로 해야 하며, 부호 있는 $\langle m\rangle$은 정확히 0이다.

**다음 노트북: 06a.** 이산 상태의 MCMC로 정상분포를 표본화하는 법까지 익혔다. 이제 시간과 상태를 모두 연속으로 보내는 극한 — 랜덤 워크에서 브라운 운동으로(모듈 06) — 로 넘어간다.

log/progress.md 한 줄 템플릿: `- YYYY-MM-DD C3 MCMC·이징 — 합격 기준 1–5 통과 (L_MAIN=…, 최대 |z|=…, tau(T_c)=…), 막힌 곳: …, 다음: 06a`